# Affine air-gap ARC-AGI-3 submit

Internet **disabled**. No online agent play.

Embeds verified local suite mastery parquet (bp35 9/9 · ar25 8/8 · ls20 7/7).
Follows top-notebook contract: columns `row_id, game_id, end_of_game, score`
written to `/kaggle/working/submission.parquet` (Goose / random-agent / simplified).

**Steward:** Run All → Save Version → Submit (after UTC daily quota reset).
Notebooks only — do not burn quota on direct file upload.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import os
import shutil
from pathlib import Path

import pandas as pd

WORKING = Path("/kaggle/working")
SRC_CANDIDATES = [
    Path("payload/submission.parquet"),
    Path("/kaggle/working/payload/submission.parquet"),
]


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()


def load_bytes() -> bytes:
    for cand in SRC_CANDIDATES:
        if cand.is_file():
            print(f"loading parquet payload: {cand}")
            return cand.read_bytes()
    b64 = Path("embedded_submission_parquet_b64.txt")
    if b64.is_file():
        print(f"loading parquet payload from {b64}")
        return base64.b64decode(b64.read_text(encoding="utf-8"))
    raise FileNotFoundError("No embedded AGI-3 submission.parquet found in kernel files.")


raw = load_bytes()
out = WORKING / "submission.parquet"
out.write_bytes(raw)

df = pd.read_parquet(out)
required = ["row_id", "game_id", "end_of_game", "score"]
assert list(df.columns) == required, list(df.columns)
assert df["end_of_game"].dtype == bool
assert str(df["score"].dtype).startswith("int")
print(df)
print(f"Wrote {out} rows={len(df)} sha256={sha256(out)}")
print(f"KAGGLE_IS_COMPETITION_RERUN={os.getenv('KAGGLE_IS_COMPETITION_RERUN')!r}")
# Air-gap mastery path: same parquet for commit + rerun (no gateway solve).
